In [ ]:
# 환경 설정 및 파일 로드
import nibabel as nib
import numpy as np
import pandas as pd
import subprocess  # DCM2NIIX 실행용
import time
import json
import os
import random
from sklearn.metrics import accuracy_score, roc_auc_score

# --- 설정값 (프로젝트에 맞춰 테스트 진행 전 수정!!!!) ---
# MRI 파일 경로 (테스트용 .dcm 또는 .nii.gz)
MRI_TEST_PATH = './test_data/subject_001.dcm'
# 모델 파일 경로
HIPPMAPP3R_MODEL_PATH = './models/HippMapp3r_final.pth' # HippMapp3r 모델 파일
XGBOOST_MODEL_PATH = './models/xgb_hippo.json' # XGBoost 모델 파일
LABEL_FILE_PATH = './data/train_index.csv' # 라벨링 정보 파일

# 전처리 후 예상되는 Numpy 배열 크기
EXPECTED_ARRAY_SHAPE = (160, 160, 160)


## 7:1:2 데이터 분할 로직 검증
print("--- [새로운 단위 테스트] 7:1:2 데이터 분할 비율 검증 시작 ---")

try:
    # 1. 라벨 파일 로드 및 전체 샘플 개수 계산
    label_df = pd.read_csv(LABEL_FILE_PATH)
    TOTAL_SAMPLES = len(label_df)

    # 2. 테스트용 임시 결과 가정 (실제 분할 함수 로직을 시뮬레이션)
    # 실제 환경에서는 여기서 데이터 분할 함수를 호출하고 리턴된 데이터셋의 크기를 사용해야 합니다.
    train_count = int(TOTAL_SAMPLES * 0.7)
    val_count = int(TOTAL_SAMPLES * 0.1)
    test_count = TOTAL_SAMPLES - train_count - val_count

    print(f"총 샘플 수: {TOTAL_SAMPLES}개")
    print(f"가정된 분할: 학습 {train_count}, 검증 {val_count}, 테스트 {test_count}")

    # 3. 비율 정확성 검증 (약간의 오차 허용 가능: 0.01 = 1%)
    assert abs(train_count / TOTAL_SAMPLES - 0.7) < 0.01, "학습 데이터 비율이 70%와 일치하지 않음"
    assert abs(test_count / TOTAL_SAMPLES - 0.2) < 0.01, "테스트 데이터 비율이 20%와 일치하지 않음"

    print(" 🙂 7:1:2 데이터 분할 비율 검증 성공")

except FileNotFoundError:
    print(f" ⚠️ 경고: {LABEL_FILE_PATH} 파일을 찾을 수 없어 데이터 분할 테스트를 건너뜁니다.")
except Exception as e:
    print(f" 😒 데이터 분할 비율 검증 실패: {e}")

print("------------------------------------------------------------------")


# MRI 파일 업로드 및 전처리
## 단위 시험 5.1-1: DICOM -> NIfTI 변환 (D_01)
# DCM2NIIX 실행 경로와 출력 경로 설정 필요
def convert_dicom_to_nifti(dcm_path):
    print(f"[{dcm_path}] 파일 NIfTI 변환 시작...")
    # 실제 시스템에서는 subprocess로 dcm2niix 실행
    # 예시: subprocess.run(['dcm2niix', '-f', '%f', dcm_path])

    nifti_output_path = dcm_path.replace('.dcm', '.nii.gz')
    # 성공 가정: NIfTI 파일이 생성되었다고 가정
    return nifti_output_path

nifti_path = convert_dicom_to_nifti(MRI_TEST_PATH)
assert os.path.exists(nifti_path) == True
print(f" 🙂 5.1-1 D_01 변환 성공: {nifti_path}")


## 단위 시험 5.2-1, 5.2-2, 5.2-3: 정규화 및 불필요 구조 제거 (NOR_01, NOR_02, NOR_03)
def run_preprocessing(nifti_path):
    # nibabel로 NIfTI 로드
    img = nib.load(nifti_path)
    data = img.get_fdata()

    # 1. 크기 정규화 및 배열 변환 (NOR_01, NOR_02)
    # 실제 정규화 코드가 여기에 들어갑니다. (리샘플링 및 크롭)
    preprocessed_data = np.zeros(EXPECTED_ARRAY_SHAPE) # 정규화된 데이터 가정

    # 2. 불필요 구조 제거 (Skull Stripping, Bias Correction 등) (NOR_03)
    # 뇌 외부 영역을 0으로 설정하여 제거했다고 가정

    return preprocessed_data

preprocessed_data = run_preprocessing(nifti_path)

# Numpy 배열 크기 정규화 확인
assert preprocessed_data.shape == EXPECTED_ARRAY_SHAPE
print(f" 🙂 5.2-1, 5.2-2 정규화 성공: 배열 크기 {preprocessed_data.shape} 일치")

# 불필요 구조 제거 확인 (간단한 배열 통계로 확인)
# 배열의 경계 부분이 0으로 채워져 불필요한 구조가 제거되었는지 육안으로 확인 필요 (추가)
print(" 5.2-3 불필요 구조 제거 모듈 실행 완료 (시각화 확인 필요)")

# 해마 분할 및 정량 피처 추출
# 단위 시험 5.3-2, 5.3-3: 해마 분할 (SEG_02, SEG_03)

def run_segmentation(data, model_path):
    # HippMapp3r 모델 로드
    # model = load_pytorch_model(model_path)

    # 모델에 전처리 데이터 입력 후 좌/우 해마 마스크 생성
    # mask_left, mask_right = model.predict(data)

    # 테스트용 임시 마스크 생성
    mask_left = np.random.randint(0, 2, size=data.shape)
    mask_right = np.random.randint(0, 2, size=data.shape)

    print(" 5.3-2 모델 로드 및 해마 자동 분할 실행 완료")
    return mask_left, mask_right


mask_L, mask_R = run_segmentation(preprocessed_data, HIPPMAPP3R_MODEL_PATH)

# 마스크 유효성 검사
assert np.sum(mask_L) > 0, "좌측 해마 마스크가 비어 있음"
assert np.sum(mask_R) > 0, "우측 해마 마스크가 비어 있음"
print(" 🙂 단위 테스트: 해마 마스크 유효성 검증 성공")


# 단위 시험 5.4-1, 5.4-2: 정량 피처 추출 (NFE_01, NFE_02)

def extract_features(mask_L, mask_R, ICV=1500000):
    """
    ICV 기본값: 1,500,000 mm^3 (참고용 평균치)
    실제 프로젝트에서는 MRI 메타데이터에서 받아야 함
    """
    voxel_volume = 1.0   # mm^3 가정

    # 부피 계산
    volume_L = np.sum(mask_L) * voxel_volume
    volume_R = np.sum(mask_R) * voxel_volume
    volume_Total = volume_L + volume_R

    # 비대칭 지수
    asymmetry_index = (volume_L - volume_R) / (volume_Total / 2)

    # ICV 보정 부피 (단위 테스트 2번 목표: 로직 정확성 검증)
    volume_normalized = volume_Total / ICV

    features = {
        'Vol_L': volume_L,
        'Vol_R': volume_R,
        'Vol_Total': volume_Total,
        'Vol_Normalized': volume_normalized,
        'Asym_Index': asymmetry_index
    }
    return features

# 피처 생성
features = extract_features(mask_L, mask_R)


# 피처 값 정확성 단위 테스트

# 1) 총 부피 = L + R (수학적 로직 검증)
assert features['Vol_Total'] == features['Vol_L'] + features['Vol_R'], \
    "총 부피 계산이 L+R과 일치하지 않음"

# 2) ICV 보정 부피 정확성 검증
expected_norm = features['Vol_Total'] / 1500000
assert abs(features['Vol_Normalized'] - expected_norm) < 1e-10, \
    "ICV 보정 부피 계산이 정확하지 않음"

# 3) 값이 정상 범위인지 (음수·NaN 방지)
assert features['Vol_Total'] > 0, "총 부피가 0 이하임"
# 이 assert는 랜덤 데이터 때문에 실패할 수 있으나, 개념적 검증임
# assert 0 < features['Vol_Normalized'] < 1, "ICV 보정 부피가 비정상 범위임"

print(f" 🙂 5.4-1, 5.4-2 피처 추출 성공: 좌 해마 부피={features['Vol_L']:.2f}")
print(f" 🙂 5.4-1 ICV 보정 부피 검증 성공: 정규화 부피={features['Vol_Normalized']:.6f}")


# 단위 시험 5.4-3: XGBoost 입력 피처 변환 (NFE_03)

xgb_input_features = pd.Series(features)
print(" 5.4-3 피처 분류 모델 입력 형식으로 변환 완료")

assert isinstance(xgb_input_features, pd.Series), "XGBoost 입력 포맷이 Pandas Series가 아님"
assert len(xgb_input_features) == 5, "피처 개수가 5개가 아님"

print(" 🙂 단위 테스트: XGBoost 입력 변환 검증 성공")


# 분류 예측 및 모델 로드
## 단위 시험 5.6-1, 5.6-2, 5.6-3: 예측 및 실시간 처리 (PRE_01, PRE_02, PRE_03)
def run_prediction(features, model_path):
    start_time = time.time()

    # XGBoost 모델 로드
    # model = xgb.Booster()
    # model.load_model(model_path)

    # 예측 수행 (테스트를 위해 임의의 결과 가정)
    prob_CN = 0.92
    prob_AD = 0.08

    end_time = time.time()
    latency = end_time - start_time

    return prob_CN, prob_AD, latency

prob_CN, prob_AD, latency = run_prediction(xgb_input_features, XGBOOST_MODEL_PATH)

# 결과 확인
assert (prob_CN + prob_AD) == 1.00
print(f" 🙂 5.6-1, 5.6-2 예측 성공: CN 확률 {prob_CN*100:.1f}%, AD 확률 {prob_AD*100:.1f}%")

# 실시간/준실시간 처리 속도 확인 (PRE_03)
# 임의의 기준 (예: 1초 미만)
assert latency < 1.0
print(f" 🙂 5.6-3 처리 속도 성공: 예측 레이턴시 {latency:.4f}초 (준실시간 만족)")

# 결과 시각화 및 저장 준비
## 단위 시험 5.7-1, 5.7-3, 5.7-5: 결과 객체 생성 및 저장 준비 (RVD_01, RVD_03, RVD_05)
def prepare_result_object(features, prob_CN, prob_AD):
    # 환자 정보 및 예측 결과를 최종 JSON/Dictionary 객체로 통합
    final_result = {
        'patient_id': '23615034',
        'prediction_cn_prob': prob_CN,
        'prediction_ad_prob': prob_AD,
        'hippo_volume_L': features['Vol_L'],
        'hippo_volume_R': features['Vol_R'],
        'analysis_date': time.strftime('%Y-%m-%d %H:%M:%S')
    }

    # 이 객체를 MySQL DB에 저장하는 함수 실행 가정
    # save_to_mysql(final_result)
    print(" 5.7-5 결과 객체 생성 및 DB 저장 함수 호출 완료")

    return final_result

final_data_object = prepare_result_object(features, prob_CN, prob_AD)

# 객체 내용 확인 (RVD_01, RVD_03의 데이터 무결성 확인)
assert final_data_object['hippo_volume_L'] > 0
print(json.dumps(final_data_object, indent=2))
print(" 🙂 5.7-1, 5.7-3 결과 객체 준비 성공 (DB 저장 및 시각화 데이터 준비 완료)")

# + 단위테스트 단계의 성능 평가

# 분류 예측 모듈 실행 속도 측정

# PRE_03: 모델은 새로운 MRI 데이터에 대해 실시간 또는 준 실시간 처리가 가능한지 확인한다.
print("--- A. 분류 예측 모듈 Latency 측정 ---")

# (이전 단계에서 정의된) run_prediction 함수 재정의 및 호출
def run_prediction_timed(features):
    # 실제 XGBoost 모델 로드 및 예측 로직을 여기에 삽입합니다.
    start_time = time.time()

    # 예측 수행 (임의의 결과 가정)
    prob_CN = random.uniform(0.7, 0.95)
    prob_AD = 1 - prob_CN

    end_time = time.time()
    latency = end_time - start_time

    return latency

# 테스트 실행
PREDICTION_LATENCY_MAX = 0.5 # 예측 모듈만 수행 시 목표 시간 (예: 0.5초)

latency = run_prediction_timed(xgb_input_features)

print(f"측정된 예측 레이턴시: {latency:.4f} 초")

# 성능 평가: 준실시간 처리 기준 충족 여부 확인
if latency < PREDICTION_LATENCY_MAX:
    print(" 🙂 성능 테스트 통과: 예측 모듈이 준실시간 처리 속도를 만족합니다.")
else:
    print(f" 😒 성능 실패: 예측 시간이 {latency:.4f}초로 목표 {PREDICTION_LATENCY_MAX}초 초과.")


# 해마 분류 모델 실행 속도 측정

# PRE_03에 기여하는 주요 모듈 시간 측정
print("\n--- B. 해마 분할 모듈 Latency 측정 ---")

def run_segmentation_timed(data):
    start_time = time.time()

    # 실제 HippMapp3r 모델 로드 및 분할 로직 삽입
    mask_left = np.random.randint(0, 2, size=data.shape)
    mask_right = np.random.randint(0, 2, size=data.shape)

    end_time = time.time()
    return end_time - start_time

# 테스트 실행 (전처리된 데이터 사용)
SEGMENTATION_LATENCY_MAX = 120.0 # 분할 모듈 목표 시간 (예: 120초, 2분)

latency_seg = run_segmentation_timed(preprocessed_data)

print(f"측정된 분할 레이턴시: {latency_seg:.2f} 초")

# 성능 평가: 준실시간 처리 기준 충족 여부 확인
if latency_seg < SEGMENTATION_LATENCY_MAX:
    print(" 🙂 성능 테스트 통과: 분할 모듈이 준실시간 처리 속도를 만족합니다.")
else:
    print(f" 😒 성능 실패: 분할 시간이 {latency_seg:.2f}초로 목표 {SEGMENTATION_LATENCY_MAX}초 초과.")


# 정량 피처 정확성 검증
# NFE_02: 추출된 해마 부피 및 비대칭 지수 등의 정량 피처가 정확한지 확인한다.
print("\n--- A. 정량 피처 정확성 검증 ---")

# **테스트 데이터의 참값(Ground Truth) 가정**
# 실제 테스트에서는 전문가가 수동 분할한 값 또는 신뢰할 수 있는 레퍼런스 값 사용
GROUND_TRUTH = {
    'Vol_L': 3200.0,
    'Vol_R': 3300.0,
    'Asym_Index': 0.03 # 비대칭 지수의 참값
}
ACCEPTABLE_TOLERANCE_VOL = 100.0 # 허용 오차 (예: ±100 mm³)
ACCEPTABLE_TOLERANCE_ASYM = 0.05 # 허용 오차 (예: ±0.05)


# 1. 부피 절대 오차 확인
volume_L_extracted = features['Vol_L']
volume_R_extracted = features['Vol_R']

if abs(volume_L_extracted - GROUND_TRUTH['Vol_L']) < ACCEPTABLE_TOLERANCE_VOL and \
   abs(volume_R_extracted - GROUND_TRUTH['Vol_R']) < ACCEPTABLE_TOLERANCE_VOL:
    print(" 🙂 품질 테스트 통과: 좌/우 해마 부피가 허용 오차 범위 내에 있습니다.")
else:
    print(" 😒 품질 테스트 실패: 좌/우 해마 부피가 허용 오차 범위를 초과합니다.")

# 2. 비대칭 지수 오차 확인
asym_extracted = features['Asym_Index']

if abs(asym_extracted - GROUND_TRUTH['Asym_Index']) < ACCEPTABLE_TOLERANCE_ASYM:
    print(" 🙂 품질 테스트 통과: 비대칭 지수 계산이 정확합니다.")
else:
    print(" 😒 품질 테스트 실패: 비대칭 지수 계산이 정확하지 않습니다.")


# 예외 및 오류 처리 검증
# D_01, NOR_03: 입력 및 전처리 과정의 안정성 검증
print("\n--- B. 예외 및 오류 처리 검증 (신뢰성) ---")

def test_invalid_file_handling():
    INVALID_PATH = './non_existent_file.dcm'

    # 1. 잘못된 파일 경로 입력 테스트 (5.1 D_01 관련)
    try:
        # convert_dicom_to_nifti(INVALID_PATH) # 이 함수가 FileNotFoundError를 발생시켜야 함
        print(f" ⚠️ 에러 시뮬레이션: {INVALID_PATH} 파일을 찾을 수 없습니다.")
        raise FileNotFoundError # 오류 발생 가정

    except FileNotFoundError:
        print(" 🙂 품질 테스트 통과: 잘못된 파일 경로에 대해 FileNotFoundError 예외를 성공적으로 포착했습니다.")
    except Exception as e:
        print(f" 😒 품질 테스트 실패: 예상치 못한 오류 발생: {e}")


def test_empty_data_handling():
    # 2. 빈 데이터 입력 테스트 (5.2 전처리 관련)
    empty_data = np.zeros((1, 1, 1)) # 비정상적으로 작은/빈 데이터

    try:
        # segmentation_module(empty_data) # 이 함수가 ValueError 또는 CustomError를 발생시켜야 함
        if empty_data.size < 1000: # 임의의 임계값
            raise ValueError("입력 데이터가 너무 작습니다.")

    except ValueError:
        print(" 🙂 품질 테스트 통과: 비정상적으로 작은 데이터에 대해 ValueError 예외를 성공적으로 포착했습니다.")
    except Exception as e:
        print(f" 😒 품질 테스트 실패: 예상치 못한 오류 발생: {e}")

test_invalid_file_handling()
test_empty_data_handling()